# 🔬 Notebook 08: VisReg + Hybrid UNETR Downstream
### Part of the BraTS 3D Volumetric Multimodal JEPA Research Suite

Fine-tunes the **same SSL-pretrained encoder as notebook 07** with `HybridUNETRDecoder3D` (native 128³/64³ stem, 30 epochs, AMP), then runs the identical battery: held-out test evaluation ($N=242$), low-data label efficiency ($1\%$–$100\%$), and OOD scanner-shift robustness.
Same-encoder comparison against 07 isolates the decoder effect. Requires notebook 01's published checkpoint dataset (fail-loud intake in §5).

> **Protocol: v2 multiregion (BraTS 2024 WT/TC/ET).** 5-class outputs (bg/NCR/ED/NET/ET); eval emits one row per `(model, region)`. Never mix these tables with v1 binary whole-tumor results.


## 1. Hardware & CUDA Environment Verification


In [ ]:
import datetime
import time

NOTEBOOK_START_TIME = time.time()
NOTEBOOK_START_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"⏱️ Session Start Time: {NOTEBOOK_START_STR}")

!nvidia-smi

import torch

print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(
        "WARNING: No GPU detected. Please navigate to Notebook Settings -> Accelerator -> GPU T4!"
    )


In [ ]:
# =========================================================================
# ⚙️ Experiment Configuration & Reproducibility Parameters
# =========================================================================

SEED = 42                 # Deterministic seed for weight init, augmentations, and data splits
NUM_WORKERS = 4          # Parallel CPU data loading workers (Kaggle T4 provides 4 vCPUs)
BATCH_SIZE = 2           # Volumetric batch size for downstream fine-tuning & evaluation
PRETRAIN_BATCH_SIZE = 16  # Self-supervised pre-training batch size

# Belt-and-suspenders: pin the mounted full-pool dataset so every script
# resolving the default name finds it even without an explicit data_dir.
import os as _os
from pathlib import Path as _Path
_kaggle_cands = [_Path("/kaggle/input/brats-3d-full/brats_gli_3d_v2"), _Path("/kaggle/input/brats-3d-full/brats_gli_3d_full")]  # v2 folder first
_kaggle_full = next((p for p in _kaggle_cands if p.is_dir() and (p / "metadata.csv").exists()), _kaggle_cands[1])
if _kaggle_full.is_dir() and (_kaggle_full / "metadata.csv").exists():
    _os.environ["BRATS3D_DATA_DIR"] = str(_kaggle_full)
    print(f"Dataset env override: BRATS3D_DATA_DIR={_kaggle_full}")


## 2. Dependencies Installation


In [ ]:
!pip install -q --no-cache-dir monai nibabel tabulate matplotlib
import monai
import nibabel as nib
import tabulate

print(f"✓ MONAI Version:    v{monai.__version__}")
print(f"✓ NiBabel Version:  v{nib.__version__}")
print(f"✓ Tabulate Version: v{tabulate.__version__}")

## 3. Codebase Setup & Editable Installation


In [ ]:
import os
import shutil
import sys
from pathlib import Path

# Setup working directory in /kaggle/working
REPO_URL = "https://github.com/hanriman/tumor_segmentation_3d.git"
work_dir = Path("/kaggle/working/tumor_segmentation_3d")

# Clean up broken or incomplete clone from previous failed runs
if work_dir.exists() and not (work_dir / "src" / "brats_jepa_3d").exists():
    print("⚠️ Cleaning up incomplete repository clone...")
    shutil.rmtree(work_dir)

thesis_repo = Path("/kaggle/working/thesis_repo")
if thesis_repo.exists() and not (thesis_repo / "src" / "brats_jepa_3d").exists():
    shutil.rmtree(thesis_repo)

# Clone repository if not already present
if not (work_dir / "src" / "brats_jepa_3d").exists():
    if (thesis_repo / "src" / "brats_jepa_3d").exists():
        work_dir = thesis_repo
    elif Path("/kaggle/working/src/brats_jepa_3d").exists():
        work_dir = Path("/kaggle/working")
    else:
        print(f"Cloning codebase from: {REPO_URL} ...")
        !git clone {REPO_URL} {work_dir}

# Verify package was successfully cloned
src_dir = work_dir / "src"
if not (src_dir / "brats_jepa_3d").exists():
    raise RuntimeError(
        "❌ Clone failed! The package 'brats_jepa_3d' was not found on disk.\n"
        "👉 Please ensure 'Internet' is toggled ON in the Kaggle notebook settings (right sidebar)!"
    )

# Change working directory and update sys.path
os.chdir(str(work_dir))
%cd {work_dir}

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Install in editable mode
!pip install -q -e .

print(f"\n✓ Working directory set to: {Path.cwd()}")
!git log -1 --oneline 2>/dev/null || echo "(Git commit info unavailable)"

# Pin the multiregion branch: v2 code (out_channels=5, regions.py, multiclass
# paths) exists ONLY here. Plain clone lands on main = silent binary-on-integer garbage.
!git -C {work_dir} checkout feat/multiregion-wt-tc-et
!git -C {work_dir} pull --ff-only origin feat/multiregion-wt-tc-et
import pathlib as _pl
assert (_pl.Path(work_dir) / "src/brats_jepa_3d/metrics/regions.py").exists(), \
    "Branch checkout failed: regions.py missing — aborting before GPU hours burn."
print("On branch feat/multiregion-wt-tc-et:", end=" ")
!git -C {work_dir} log -1 --oneline


## 4. Dataset Discovery & Health Checks
Verifies whether processed `.npz` volumes exist. If absent, automatically invokes `prepare_data_3d.py` with multi-worker parallel resampling (`--num_workers 4`) and compact `float16` storage (~6.7 MB per volume, upcast to FP32 in RAM upon loading).


In [ ]:
import pandas as pd
import torch

from brats_jepa_3d.config import get_dataset_dir, get_metadata_path
from brats_jepa_3d.data import BraTS3DDataset

print("=== DATASET DISCOVERY ===")
data_dir = get_dataset_dir("brats_gli_3d_full")
meta_path = get_metadata_path("brats_gli_3d_full")
print(f"Dataset Directory: {data_dir} (Exists: {data_dir.exists()})")
print(f"Metadata CSV:      {meta_path} (Exists: {meta_path.exists()})")

# If preprocessed dataset is not found, check for raw BraTS data and run prepare_data_3d.py
if not meta_path.exists():
    print("\n⚠️ Preprocessed dataset not found. Running 3D preprocessing from raw BraTS data...")
    !python scripts/prepare_data_3d.py --limit 100 --dtype float16 --num_workers {NUM_WORKERS} --seed {SEED}
    meta_path = get_metadata_path("brats_gli_3d_full")

if meta_path.exists():
    df = pd.read_csv(meta_path)
    print(f"\n✓ Loaded Metadata: {len(df)} total records across splits:")
    print(df["split"].value_counts().to_string())

    ds = BraTS3DDataset(data_dir=data_dir, split="train")
    sample = ds[0]
    print("\n✓ Sample Tensor Verification:")
    print(f"  Image Shape: {sample['image'].shape} (dtype: {sample['image'].dtype})")
    print(f"  Mask Shape:  {sample['mask'].shape} (dtype: {sample['mask'].dtype})")
    print(f"  Tumor Voxels: {int((sample['mask'] > 0).sum()):,}")
else:
    print(
        "❌ ERROR: Please attach 'brats-3d-full' or raw BraTS dataset via '+ Add Input' in Kaggle!"
    ) --label_protocol multiregion
print("Required attachment: brats-3d-full dataset VERSION 2 (integer masks 0-4, has_et column). Attachments pin versions — a stale v1 pin silently mounts binary masks.")

# v2 protocol guard: fail loud on stale v1 binary attachment
assert "has_et" in df.columns, "metadata.csv lacks has_et — stale v1 dataset pin; attach brats-3d-full VERSION 2"
import numpy as _np
_vals = set(_np.unique(sample["mask"].numpy()).tolist())
assert _vals <= {0.0, 1.0, 2.0, 3.0, 4.0} and _vals != {0.0, 1.0}, f"mask values {_vals} look like v1 binary; attach v2 integer masks"


## 5. Pre-trained Checkpoint Intake (fail loud)
Copies the SSL-pretrained encoder from notebook 01 — attached as a Kaggle dataset input — into `outputs/checkpoints/`. Aborts when absent: proceeding without it would silently benchmark random initialization.


In [ ]:
# Checkpoint intake: SSL-pretrained encoder from notebook 01 (published as a
# Kaggle dataset input). Fail LOUD when absent: train_downstream_3d.py would
# otherwise warn once and silently train from random initialization,
# invalidating the SSL comparison.
import os
import shutil
from pathlib import Path

from brats_jepa_3d.config import CHECKPOINTS_DIR
from brats_jepa_3d.utils import sort_checkpoints_by_epoch

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
# Mount root is env-overridable (BRATS3D_CKPT_MOUNT) so intake is testable
# off-Kaggle; defaults to the Kaggle input mount.
ckpt_mount = Path(os.environ.get('BRATS3D_CKPT_MOUNT', '/kaggle/input'))


def _is_pretrain_ckpt(p: Path) -> bool:
    n = p.name.lower()
    return n.startswith("visreg_jepa") and "scratch" not in n and not any(
        d in n for d in ("multiscale", "bottleneck", "unetr_hybrid", "hybrid", "downstream")
    )


cands = [p for p in sorted(CHECKPOINTS_DIR.glob("visreg_jepa*_3d_best.pt")) if _is_pretrain_ckpt(p)]
if not cands and ckpt_mount.exists():
    for src in sorted(ckpt_mount.rglob("visreg_jepa*_3d_best.pt")):
        if _is_pretrain_ckpt(src):
            dest = CHECKPOINTS_DIR / src.name
            if not dest.exists():
                shutil.copy2(src, dest)
            cands.append(dest)
            break
if not cands:
    pretrain_epochs = [
        p for p in sort_checkpoints_by_epoch(list(CHECKPOINTS_DIR.glob("visreg_jepa*epoch*.pt")))
        if _is_pretrain_ckpt(p)
    ]
    cands = pretrain_epochs[-1:] if pretrain_epochs else []
assert cands, (
    "No SSL pretrain checkpoint (visreg_jepa*_3d_best.pt) in outputs/checkpoints/ or "
    "/kaggle/input/. Attach notebook 01's published checkpoint dataset first — "
    "refusing to train from random init."
)
pretrained_ckpt = str(cands[-1])
print(f"Using pre-trained checkpoint: {pretrained_ckpt}")


## 7b. Hybrid UNETR Decoder (Native 128³/64³ Skips)
Fine-tunes the **same brain-aware pre-trained encoder** with `HybridUNETRDecoder3D` (`--decoder_type unetr_hybrid`): FPN stages 1–2 transfer verbatim, only `fuse3/fuse4` re-initialize and the narrow conv stem trains fresh. Compares directly against the §7 multiscale result.


In [ ]:
!python scripts/train_downstream_3d.py \
    --model_type visreg_jepa \
    --decoder_type unetr_hybrid \
    --pretrained_checkpoint {pretrained_ckpt} \
    --seed {SEED} \
    --epochs 30 \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --learning_rate 3e-4 \
    --deep_supervision \
    --amp


## 7. Full-Data Held-Out Test Split Evaluation (hybrid decoder)
Evaluates the hybrid-decoder VisReg model across all $242$ held-out test volumes.


In [ ]:
!python scripts/evaluate_3d.py \
    --model_type visreg_jepa \
    --decoder_type unetr_hybrid \
    --seed {SEED} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --amp \
    --tta


## 8. Low-Data Volumetric Label Efficiency (hybrid decoder)
Same six annotation budgets as notebook 07 ($1\%$–$100\%$): isolates the decoder's contribution to label efficiency on the identical SSL encoder.


In [ ]:
!python scripts/evaluate_low_data_3d.py \
    --model_type visreg_jepa \
    --decoder_type unetr_hybrid \
    --fractions 0.01 0.05 0.10 0.25 0.50 1.00 \
    --seed {SEED} \
    --epochs 15 \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --amp


## 9. Out-of-Distribution (OOD) Scanner Shift Robustness (hybrid decoder)
Evaluates the hybrid-decoder 3D VisReg JEPA under physical 3D Rician scanner noise ($\sigma=0.08$), quadratic RF $B_1$ field bias, and missing sequence triage (T1c-only, FLAIR-only) — the regime where native high-res skips should help most. $B_1$ fields are per-sample random; Rician/B1 tissue models are sign-preserving (Rician-if-nonnegative / Gaussian-if-negative, gain on original voxels). OOD tables measured before the sign-preserving fix are **not comparable** — re-run evaluate_ood_3d.py; see README §6.3.


In [ ]:
!python scripts/evaluate_ood_3d.py \
    --model_type visreg_jepa \
    --decoder_type unetr_hybrid \
    --seed {SEED} \
    --batch_size 1 \
    --num_workers {NUM_WORKERS} \
    --amp


## 10. Export & Package Artifacts
Packages hybrid-finetuned weights, training logs, and metric reports into `visreg_unetr_outputs.zip` for 1-click download.


In [ ]:
import datetime
import time
from pathlib import Path

from brats_jepa_3d.config import IN_KAGGLE, PROJECT_ROOT
from brats_jepa_3d.utils import export_artifacts

# 1. Package hybrid-finetuned 3D VisReg checkpoints, logs, and metric reports
base_working = Path("/kaggle/working") if IN_KAGGLE else PROJECT_ROOT
export_res = export_artifacts(
    export_name="visreg_unetr_outputs",
    export_dir=base_working / "export_visreg_unetr",
    zip_path=base_working / "visreg_unetr_outputs.zip",
    model_prefix="unetr",
    verbose=True,
)

# 2. Session Timing Report
NOTEBOOK_END_TIME = time.time()
NOTEBOOK_END_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
start_time = globals().get("NOTEBOOK_START_TIME", NOTEBOOK_END_TIME)
total_elapsed_sec = NOTEBOOK_END_TIME - start_time
hours, rem = divmod(total_elapsed_sec, 3600)
minutes, seconds = divmod(rem, 60)

print("\n" + "=" * 50)
print(f"⏱️ Session Start Time:   {globals().get('NOTEBOOK_START_STR', 'N/A')}")
print(f"⏱️ Session End Time:     {NOTEBOOK_END_STR}")
print(f"⏱️ Total Execution Time: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({total_elapsed_sec:.2f}s)")
print("=" * 50)


In [ ]:
import time
print("Keep-alive on: press Enter to keep going, type 'q' + Enter (or Stop) to end.")
while True:
    try:
        s = input("keep-alive — Enter to continue, q to quit: ").strip().lower()
    except EOFError:
        time.sleep(300)
        print('…still alive', flush=True)
        continue
    except KeyboardInterrupt:
        print('Stopped.')
        break
    if s == 'q':
        print('Ending keep-alive.')
        break
